In [5]:
import gc
import sys
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm

# ── 데이터 ──
MEAN = [0.48235, 0.45882, 0.40784]
STD  = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.CenterCrop(56),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
dataset = ImageFolder('./data', transform=transform)
small_train = Subset(dataset, range(100))
small_test  = Subset(dataset, range(100, 120))
train_loader = DataLoader(small_train, batch_size=4, shuffle=True,  num_workers=0)
test_loader  = DataLoader(small_test,  batch_size=4, shuffle=False, num_workers=0)

# ── gc 확인 ──
print(f'gc enabled: {gc.isenabled()}')
gc.collect()
print('gc OK')

# ── 모델 임포트 ──
sys.path.insert(0, '/home/yulee23/research')
from transformer_comparison import Spikformer, LIFNode, TTFSEncoder, reset_lif, CFG

# ── 모델 생성 ──
model = Spikformer(
    img_size   = 56,
    embed_dim  = 64,   # 경량화
    depth      = 1,    # 경량화
    num_heads  = 2,
    mlp_ratio  = 2,
    num_classes= 2,
    T          = 4,
)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

# ── 1에포크 ──
crit = nn.CrossEntropyLoss()
opt  = optim.AdamW(model.parameters(), lr=1e-3)

model.train()
total_loss, correct, total = 0., 0, 0
for imgs, labels in tqdm(train_loader, desc='train'):
    for m in model.modules():
        if isinstance(m, LIFNode): m.reset()
    opt.zero_grad()
    out  = model(imgs)
    loss = crit(out, labels)
    loss.backward()
    opt.step()
    total_loss += loss.item()
    correct    += (out.argmax(1) == labels).sum().item()
    total      += labels.size(0)
    gc.collect()

print(f'train loss={total_loss/len(train_loader):.3f} acc={correct/total*100:.1f}%')

# ── eval ──
model.eval()
total_loss, correct, total = 0., 0, 0
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='eval'):
        for m in model.modules():
            if isinstance(m, LIFNode): m.reset()
        out  = model(imgs)
        loss = crit(out, labels)
        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)

gc.collect()
print(f'val   loss={total_loss/len(test_loader):.3f} acc={correct/total*100:.1f}%')
print('완료')

gc enabled: True
gc OK
params: 53,474


train: 100%|████████████████████████████████████████████████████████████████████████████| 25/25 [00:08<00:00,  3.11it/s]


train loss=0.033 acc=98.0%


eval: 100%|███████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 12.48it/s]

val   loss=0.000 acc=100.0%
완료
